In [ ]:
import os
import sys
import time

!{sys.executable} --version
if sys.version_info.minor == 8:
    raise RuntimeError('USE JUPYTER KERNEL VENV 3.10/310/DEFAULT INSTEAD')

!cd /workspace/CRYPTO_MACAQUES && pip install .
!cd /home/crypto/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && source venv/bin/activate && !{sys.executable} -m pip install .

from IPython.core.display_functions import clear_output
clear_output()

In [ ]:
%load_ext autoreload
%autoreload

from datetime import timedelta
from dateutil import parser

from SRC.LIBRARIES.new_data_utils import fetch
from SRC.CORE._CONSTANTS import _KIEV_TIMESTAMP
import SRC.LIBRARIES.new_utils as nu


SYM = 'XRP'.upper()
SYMBOL = f'{SYM}USDT'
DISCRETIZATION = '1d'.upper()
DISPLAY_START_DATE_STR = '2023-01-01'
DISPLAY_END_DATE_STR = '2026-05-21'
CAPITAL_PER_TRADE = 1000
COMMISSION_RATE = 0.00075
SLIPPAGE = 0.0002
TRAILING_TP = True
TRAILING_TP_DISTANCE = 0.5
FORCE_CLOSE_AT_END = False

# ========== КРАСНАЯ СТРАТЕГИЯ ==========
RED_TP_LEVEL = 'yellow'           # near_green, yellow, far_green
RED_TP_MODE = 'dynamic'               # fixed, dynamic
RED_SL_RATIO = 1
RED_REQUIRE_GREEN_TOUCH_AFTER_SL = False
RED_MIN_BARS_OUTSIDE = 1
RED_STOCH_FILTER = False               # включить/выключить фильтр
RED_STOCH_SERIES = 'K'                 # 'K' или 'D' - какой стохастик использовать
RED_STOCH_OVERSOLD = 30                # нижняя граница для long
RED_STOCH_OVERBOUGHT = 70              # верхняя граница для short

# ========== ЗЕЛЁНАЯ СТРАТЕГИЯ ==========
ENABLE_GREEN_STRATEGY = True                # включить/выключить зелёную стратегию
GREEN_TP_LEVEL = 'green'                    # green, yellow
GREEN_TP_MODE = 'dynamic'                   # fixed, dynamic
GREEN_REQUIRE_YELLOW_TOUCH_AFTER_SL = False # касание жёлтой линии после стопа
GREEN_MIN_BARS_OUTSIDE = 1
GREEN_STOCH_FILTER = False
GREEN_STOCH_SERIES = 'K'                   # 'K' или 'D'
GREEN_STOCH_OVERSOLD = 20
GREEN_STOCH_OVERBOUGHT = 80

market_type = 'SPOT'
mrc_buffer = 1000
rsi_buffer = 200
stoch_buffer = 50
macd_buffer = 200
atr_buffer = 200
display_start_dt = parser.parse(DISPLAY_START_DATE_STR)
load_end_date = parser.parse(DISPLAY_END_DATE_STR)
start_time = time.perf_counter()

discretization_value = int(DISCRETIZATION[:-1])
timeframe_seconds = {
    'M': discretization_value * 60,
    'H': discretization_value * 3600,
    'D': discretization_value * 86400
}.get(DISCRETIZATION[-1], 1800)

load_start_dt = display_start_dt - timedelta(seconds=timeframe_seconds * mrc_buffer)

mrc_df = fetch(market_type, SYMBOL, DISCRETIZATION, load_start_dt, load_end_date)
df = mrc_df.iloc[mrc_buffer:].copy()

In [ ]:
# Убеждаемся, что индексы уникальны
mrc_df = mrc_df[~mrc_df.index.duplicated(keep='first')]
df = df[~df.index.duplicated(keep='first')]

# 2. Stochastic
stoch_calc_df = nu.prepare_buffer_data(mrc_df, df, stoch_buffer)
df = nu.stochastic_tradingview(df, stoch_calc_df)

# Расчёт MRC
mrc_df = nu.mrc_calculate(mrc_df, df)

df_counter = mrc_df.iloc[mrc_buffer - 1 - max(RED_MIN_BARS_OUTSIDE, GREEN_MIN_BARS_OUTSIDE):].copy()

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List


# ========== ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ==========

def get_metrics(total_pnl, avg_return_per_trade_percent, win_rate, num_trades, avg_win, avg_loss):
    return {
        'total_pnl': total_pnl,
        'avg_return_per_trade_percent': avg_return_per_trade_percent,
        'win_rate': win_rate,
        'num_trades': num_trades,
        'avg_win': avg_win,
        'avg_loss': avg_loss
    }


def calculate_metrics(trades: List[Dict], capital_per_trade: float) -> Dict:
    if len(trades) == 0:
        return get_metrics(0, 0, 0, 0, 0, 0)

    trades_df = pd.DataFrame(trades)
    total_pnl = trades_df['pnl'].sum()
    avg_return_per_trade_percent = (total_pnl / (capital_per_trade * len(trades_df))) * 100
    win_rate = (trades_df['pnl'] > 0).mean()
    num_trades = len(trades_df)
    positive_trades = trades_df['pnl'] > 0
    negative_trades = trades_df['pnl'] < 0
    avg_win = trades_df[positive_trades]['pnl'].mean() if positive_trades.any() else 0.0
    avg_loss = trades_df[negative_trades]['pnl'].mean() if negative_trades.any() else 0.0

    return get_metrics(total_pnl=total_pnl, avg_return_per_trade_percent=avg_return_per_trade_percent, win_rate=win_rate, num_trades=num_trades, avg_win=avg_win, avg_loss=avg_loss)


def calculate_pnl(position: Dict, exit_price: float, slippage: float, commission: float, capital_per_trade: float) -> float:
    """Расчёт PnL для закрытой позиции"""
    if position['type'] == 'long':
        exit_price_adj = exit_price * (1 - slippage)
        exit_proceeds = position['amount'] * exit_price_adj * (1 - commission)
        return exit_proceeds - capital_per_trade
    else:
        exit_price_adj = exit_price * (1 + slippage)
        entry_proceeds = position['amount'] * position['entry_price'] * (1 - commission)
        exit_cost = position['amount'] * exit_price_adj * (1 + commission)
        return entry_proceeds - exit_cost


def update_trailing_fixed(position: Dict, current_high: float, current_low: float, trailing_distance: float):
    """Обновление трейлинга для fixed режима"""
    if position['type'] == 'long':
        if current_high > position['tp_highest_price']:
            position['tp_highest_price'] = current_high
            new_tp = position['tp_highest_price'] * (1 - trailing_distance / 100)
            if new_tp > position['initial_tp']:
                position['tp'] = max(position['tp'], new_tp)
    else:
        if current_low < position['tp_lowest_price']:
            position['tp_lowest_price'] = current_low
            new_tp = position['tp_lowest_price'] * (1 + trailing_distance / 100)
            if new_tp < position['initial_tp']:
                position['tp'] = min(position['tp'], new_tp)


def force_close_position(position: Dict, df_back: pd.DataFrame, slippage: float, commission: float, capital_per_trade: float) -> Dict:
    """Принудительное закрытие позиции по последней цене"""
    last_price = df_back.iloc[-1]['close']

    if position['type'] == 'long':
        exit_price_adj = last_price * (1 - slippage)
        exit_proceeds = position['amount'] * exit_price_adj * (1 - commission)
        pnl = exit_proceeds - capital_per_trade
    else:
        exit_price_adj = last_price * (1 + slippage)
        entry_proceeds = position['amount'] * position['entry_price'] * (1 - commission)
        exit_cost = position['amount'] * exit_price_adj * (1 + commission)
        pnl = entry_proceeds - exit_cost

    return {
        'strategy': position['strategy'],
        'type': position['type'],
        'entry_idx': position['entry_idx'],
        'exit_idx': len(df_back) - 1,
        'entry_price': position['entry_price'],
        'exit_price': exit_price_adj,
        'pnl': pnl,
        'exit_type': 'forced'
    }


def select_stoch_values(df_back: pd.DataFrame, stoch_series: str) -> pd.Series:
    """Выбор серии стохастика (K или D)"""
    if stoch_series.upper() == 'K':
        return df_back['stoch_k']
    else:
        return df_back['stoch_d']


def get_df_back(df):
    return df.copy()


def get_enter(df_back, df_counter, band, condition_series, min_bars_outside):
    candle_bodies_outside_band_counter = 0
    len_condition_series = len(condition_series)
    candle_bodies_outside_band = np.zeros(len_condition_series)

    for i in range(len_condition_series):
        if condition_series.iloc[i]:
            candle_bodies_outside_band_counter += 1
        else:
            candle_bodies_outside_band_counter = 0

        candle_bodies_outside_band[i] = candle_bodies_outside_band_counter

    count_series = pd.Series(candle_bodies_outside_band, index=condition_series.index)
    previous_candle_bodies_outside_band_count = np.roll(count_series, 1)
    previous_candle_bodies_outside_band_count = previous_candle_bodies_outside_band_count[len(df_counter) - len(df_back):]
    candle_touches_band = ((df_back['low'] <= df_back[band]) & (df_back['high'] >= df_back[band])).values

    return candle_touches_band & (previous_candle_bodies_outside_band_count >= min_bars_outside)


def get_long_enter(df_back, df_counter, loband, min_bars_outside):
    return pd.Series(
        get_enter(df_back, df_counter, loband, (df_counter['open'] < df_counter[loband]) | (df_counter['close'] < df_counter[loband]), min_bars_outside),
        index=df_back.index
    )


def get_short_enter(df_back, df_counter, upband, min_bars_outside):
    return pd.Series(
        get_enter(df_back, df_counter, upband, (df_counter['open'] > df_counter[upband]) | (df_counter['close'] > df_counter[upband]), min_bars_outside),
        index=df_back.index
    )


def get_db_back_range(df_back):
    return range(0, len(df_back))


def get_df_back_high(df_back, i):
    return df_back.iloc[i]['high']


def get_df_back_low(df_back, i):
    return df_back.iloc[i]['low']


def get_loband1(df_back, i):
    return df_back.iloc[i]['loband1']


def get_upband1(df_back, i):
    return df_back.iloc[i]['upband1']


def get_loband2(df_back, i):
    return df_back.iloc[i]['loband2']


def get_upband2(df_back, i):
    return df_back.iloc[i]['upband2']


def get_meanline(df_back, i):
    return df_back.iloc[i]['meanline']


def get_current_stoch(stoch_values, i):
    return stoch_values.iloc[i]


def get_is_long_signal(long_enter, i):
    return long_enter.iloc[i]


def get_is_short_signal(short_enter, i):
    return short_enter.iloc[i]


def get_can_enter_by_stoch_filter(stoch_filter, direction, current_stoch, stoch_oversold, stoch_overbought, can_enter):
    if stoch_filter:
        if direction == 'long':
            if not (current_stoch < stoch_oversold):
                can_enter = False
        elif direction == 'short':
            if not (current_stoch > stoch_overbought):
                can_enter = False

    return can_enter


def get_enter_amount(commission, entry_price):
    return (CAPITAL_PER_TRADE * (1 - commission)) / entry_price


def get_long_position(strategy, i, entry_price, amount, tp_price, sl_price, trailing_tp, tp_line, tp_mode):
    return {
        'type': 'long',
        'strategy': strategy,
        'entry_idx': i,
        'entry_price': entry_price,
        'amount': amount,
        'tp': tp_price,
        'initial_tp': tp_price,
        'sl': sl_price,
        'initial_sl': sl_price,
        'tp_trailing_active': trailing_tp,
        'tp_highest_price': entry_price,
        'tp_line': tp_line,
        'tp_mode': tp_mode,
        'trailing_activated': False
    }


def get_short_position(strategy, i, entry_price, amount, tp_price, sl_price, trailing_tp, tp_line, tp_mode):
    return {
        'type': 'short',
        'strategy': strategy,
        'entry_idx': i,
        'entry_price': entry_price,
        'amount': amount,
        'tp': tp_price,
        'initial_tp': tp_price,
        'sl': sl_price,
        'initial_sl': sl_price,
        'tp_trailing_active': trailing_tp,
        'tp_lowest_price': entry_price,
        'tp_line': tp_line,
        'tp_mode': tp_mode,
        'trailing_activated': False
    }


def get_red_base_dynamic_tp(position, upband1, meanline, loband1):
    if position['type'] == 'long':
        if position['tp_line'] == 'upband1':
             return upband1
        elif position['tp_line'] == 'meanline':
             return meanline
        else:
             return loband1
    else:  # short
        if position['tp_line'] == 'loband1':
            return loband1
        elif position['tp_line'] == 'meanline':
            return meanline
        else:
            return upband1


def get_green_base_dynamic_tp(position, upband1, meanline, loband1):
    if position['type'] == 'long':
        if position['tp_line'] == 'upband1':
            return upband1
        else:  # meanline
            return meanline
    else:  # short
        if position['tp_line'] == 'loband1':
            return loband1
        else:  # meanline
            return meanline


def exit_from_position(strategy, position, upband1, meanline, loband1, current_high, current_low, trailing_tp_distance, slippage, commission, trades, i):
     # ========== ВЫХОД ИЗ ПОЗИЦИИ ==========
    if position is not None:

        # Сохраняем базовый динамический TP до возможных изменений
        base_dynamic_tp = None

        # Динамическое обновление TP
        if position['tp_mode'] == 'dynamic':
            if strategy == 'red':
                base_dynamic_tp = get_red_base_dynamic_tp(position, upband1, meanline, loband1)
            else:
                base_dynamic_tp = get_green_base_dynamic_tp(position, upband1, meanline, loband1)

            # Если трейлинг ещё не активирован, используем базовый динамический TP
            if not position.get('trailing_activated', False):
                position['tp'] = base_dynamic_tp

        # ========== ТРЕЙЛИНГ ДЛЯ FIXED РЕЖИМА ==========
        if position['tp_mode'] == 'fixed' and position['tp_trailing_active']:
            update_trailing_fixed(position, current_high, current_low, trailing_tp_distance)

        # ========== ТРЕЙЛИНГ ДЛЯ DYNAMIC РЕЖИМА (АКТИВАЦИЯ) ==========
        if (position['tp_mode'] == 'dynamic' and
            position['tp_trailing_active'] and
            not position.get('trailing_activated', False)):

            if position['type'] == 'long':
                if current_high >= base_dynamic_tp:
                    position['trailing_activated'] = True
                    position['tp_highest_price'] = max(position['tp_highest_price'], current_high)
                    new_tp = position['tp_highest_price'] * (1 - trailing_tp_distance / 100)
                    position['tp'] = max(base_dynamic_tp, new_tp)
            else:  # short
                if current_low <= base_dynamic_tp:
                    position['trailing_activated'] = True
                    position['tp_lowest_price'] = min(position['tp_lowest_price'], current_low)
                    new_tp = position['tp_lowest_price'] * (1 + trailing_tp_distance / 100)
                    position['tp'] = min(base_dynamic_tp, new_tp)

        # ========== ОБНОВЛЕНИЕ ТРЕЙЛИНГА ПОСЛЕ АКТИВАЦИИ (DYNAMIC) ==========
        if (position['tp_mode'] == 'dynamic' and
            position.get('trailing_activated', False) and
            position['tp_trailing_active']):

            if position['type'] == 'long':
                if current_high > position['tp_highest_price']:
                    position['tp_highest_price'] = current_high
                    new_tp = position['tp_highest_price'] * (1 - trailing_tp_distance / 100)
                    current_base_tp = base_dynamic_tp if base_dynamic_tp is not None else position['initial_tp']
                    position['tp'] = max(current_base_tp, new_tp)
            else:  # short
                if current_low < position['tp_lowest_price']:
                    position['tp_lowest_price'] = current_low
                    new_tp = position['tp_lowest_price'] * (1 + trailing_tp_distance / 100)
                    current_base_tp = base_dynamic_tp if base_dynamic_tp is not None else position['initial_tp']
                    position['tp'] = min(current_base_tp, new_tp)

        # ========== ПРОВЕРКА УСЛОВИЙ ВЫХОДА ==========
        exit_price = None
        exit_type = None

        if position['type'] == 'long':
            if current_high >= position['tp']:
                exit_price = position['tp']
                exit_type = 'tp_trailing' if position.get('trailing_activated', False) or (position['tp_mode'] == 'fixed' and position['tp_trailing_active']) else 'tp'
            elif current_low <= position['sl']:
                exit_price = position['sl']
                exit_type = 'sl'

        else:  # short
            if current_low <= position['tp']:
                exit_price = position['tp']
                exit_type = 'tp_trailing' if position.get('trailing_activated', False) or (position['tp_mode'] == 'fixed' and position['tp_trailing_active']) else 'tp'
            elif current_high >= position['sl']:
                exit_price = position['sl']
                exit_type = 'sl'

        if exit_price is not None:
            pnl = calculate_pnl(position, exit_price, slippage, commission, CAPITAL_PER_TRADE)

            trades.append({
                'strategy': position['strategy'],
                'type': position['type'],
                'entry_idx': position['entry_idx'],
                'exit_idx': i,
                'entry_price': position['entry_price'],
                'exit_price': exit_price,
                'pnl': pnl,
                'exit_type': exit_type
            })

            position = None

    return position, trades

# ========== КРАСНАЯ СТРАТЕГИЯ ==========

def backtest_red_cross(
    df: pd.DataFrame,
    df_counter: pd.DataFrame,
    commission: float = 0.00075,
    capital_per_trade: float = 1000.0,
    slippage: float = 0.0002,
    tp_level: str = 'far_green',
    tp_mode: str = 'fixed',
    trailing_tp: bool = False,
    trailing_tp_distance: float = 0.5,
    sl_ratio: float = 3.0,
    require_green_touch_after_sl: bool = False,
    min_bars_outside: int = 0,
    stoch_filter: bool = False,
    stoch_series: str = 'D',
    stoch_oversold: int = 20,
    stoch_overbought: int = 80,
    force_close_at_end: bool = False
) -> List[Dict]:
    loband = 'loband2'
    upband = 'upband2'
    df_back = get_df_back(df)
    long_enter = get_long_enter(df_back, df_counter, loband, min_bars_outside)
    short_enter = get_short_enter(df_back, df_counter, upband, min_bars_outside)
    stoch_values = select_stoch_values(df_back, stoch_series)
    position = None
    trades = []

    for i in get_db_back_range(df_back):
        current_high = get_df_back_high(df_back, i)
        current_low = get_df_back_low(df_back, i)
        loband1 = get_loband1(df_back, i)
        upband1 = get_upband1(df_back, i)
        loband2 = get_loband2(df_back, i)
        upband2 = get_upband2(df_back, i)
        meanline = get_meanline(df_back, i)
        current_stoch = get_current_stoch(stoch_values, i)
        is_long_signal = get_is_long_signal(long_enter, i)
        is_short_signal = get_is_short_signal(short_enter, i)

        if position is None:
            direction = None
            entry_price = None
            tp_price = None
            tp_line = None

            if is_long_signal:
                direction = 'long'
                entry_price = loband2 * (1 + slippage)

                if tp_level == 'near_green':
                    tp_price = loband1
                    tp_line = 'loband1'
                elif tp_level == 'yellow':
                    tp_price = meanline
                    tp_line = 'meanline'
                else:  # far_green
                    tp_price = upband1
                    tp_line = 'upband1'

            elif is_short_signal:
                direction = 'short'
                entry_price = upband2 * (1 - slippage)

                if tp_level == 'near_green':
                    tp_price = upband1
                    tp_line = 'upband1'
                elif tp_level == 'yellow':
                    tp_price = meanline
                    tp_line = 'meanline'
                else:  # far_green
                    tp_price = loband1
                    tp_line = 'loband1'

            if direction is not None:
                can_enter = True
                can_enter = get_can_enter_by_stoch_filter(stoch_filter, direction, current_stoch, stoch_oversold, stoch_overbought, can_enter)

                # Проверка касания зелёной линии после стоп-лосса
                if can_enter and require_green_touch_after_sl and len(trades) > 0:
                    last_trade = trades[-1]
                    last_exit_idx = last_trade['exit_idx']
                    last_exit_type = last_trade['exit_type']

                    if last_exit_type == 'sl':
                        touched_green = False

                        for j in range(last_exit_idx + 1, i + 1):
                            low = df_back.iloc[j]['low']
                            high = df_back.iloc[j]['high']
                            loband1_j = df_back.iloc[j]['loband1']
                            upband1_j = df_back.iloc[j]['upband1']

                            if (low <= loband1_j <= high) or (low <= upband1_j <= high):
                                touched_green = True
                                break

                        if not touched_green:
                            can_enter = False

                if can_enter:
                    amount = get_enter_amount(commission, entry_price)

                    if direction == 'long':
                        distance_to_tp = tp_price - entry_price
                        sl_price = entry_price - distance_to_tp / sl_ratio
                        position = get_long_position('red', i, entry_price, amount, tp_price, sl_price, trailing_tp, tp_line, tp_mode)

                    else:  # short
                        distance_to_tp = entry_price - tp_price
                        sl_price = entry_price + distance_to_tp / sl_ratio
                        position = get_short_position('red', i, entry_price, amount, tp_price, sl_price, trailing_tp, tp_line, tp_mode)

                    continue

        position, trades = exit_from_position('red', position, upband1, meanline, loband1, current_high, current_low, trailing_tp_distance, slippage, commission, trades, i)

    # ========== ПРИНУДИТЕЛЬНОЕ ЗАКРЫТИЕ ==========
    if position is not None and force_close_at_end:
        trades.append(force_close_position(position, df_back, slippage, commission, capital_per_trade))

    return trades


# ========== ЗЕЛЁНАЯ СТРАТЕГИЯ ==========

def backtest_green_cross(
    df: pd.DataFrame,
    df_counter: pd.DataFrame,
    commission: float = 0.00075,
    capital_per_trade: float = 1000.0,
    slippage: float = 0.0002,
    tp_level: str = 'green',
    tp_mode: str = 'dynamic',
    trailing_tp: bool = False,
    trailing_tp_distance: float = 0.5,
    require_yellow_touch_after_sl: bool = False,
    min_bars_outside: int = 0,
    stoch_filter: bool = False,
    stoch_series: str = 'D',
    stoch_oversold: int = 20,
    stoch_overbought: int = 80,
    force_close_at_end: bool = False
) -> List[Dict]:
    loband = 'loband1'
    upband = 'upband1'
    df_back = get_df_back(df)
    long_enter = get_long_enter(df_back, df_counter, loband, min_bars_outside)
    short_enter = get_short_enter(df_back, df_counter, upband, min_bars_outside)
    stoch_values = select_stoch_values(df_back, stoch_series)
    position = None
    trades = []

    for i in get_db_back_range(df_back):
        current_high = get_df_back_high(df_back, i)
        current_low = get_df_back_low(df_back, i)
        loband1 = get_loband1(df_back, i)
        upband1 = get_upband1(df_back, i)
        loband2 = get_loband2(df_back, i)
        upband2 = get_upband2(df_back, i)
        meanline = get_meanline(df_back, i)
        current_stoch = get_current_stoch(stoch_values, i)
        is_long_signal = get_is_long_signal(long_enter, i)
        is_short_signal = get_is_short_signal(short_enter, i)

        if position is None:
            direction = None
            entry_price = None
            tp_price = None
            tp_line = None

            if is_long_signal:
                direction = 'long'
                entry_price = loband1 * (1 + slippage)

                if tp_level == 'green':
                    tp_price = upband1
                    tp_line = 'upband1'
                else:  # yellow
                    tp_price = meanline
                    tp_line = 'meanline'

            elif is_short_signal:
                direction = 'short'
                entry_price = upband1 * (1 - slippage)

                if tp_level == 'green':
                    tp_price = loband1
                    tp_line = 'loband1'
                else:  # yellow
                    tp_price = meanline
                    tp_line = 'meanline'

            if direction is not None:
                can_enter = True
                can_enter = get_can_enter_by_stoch_filter(stoch_filter, direction, current_stoch, stoch_oversold, stoch_overbought, can_enter)

                # Проверка касания жёлтой линии после стоп-лосса
                if can_enter and require_yellow_touch_after_sl and len(trades) > 0:
                    last_trade = trades[-1]
                    last_exit_idx = last_trade['exit_idx']
                    last_exit_type = last_trade['exit_type']

                    if last_exit_type == 'sl':
                        touched_yellow = False

                        for j in range(last_exit_idx + 1, i + 1):
                            low = df_back.iloc[j]['low']
                            high = df_back.iloc[j]['high']
                            meanline_j = df_back.iloc[j]['meanline']

                            if low <= meanline_j <= high:
                                touched_yellow = True
                                break

                        if not touched_yellow:
                            can_enter = False

                if can_enter:
                    amount = get_enter_amount(commission, entry_price)

                    if direction == 'long':
                        # SL на красной линии
                        sl_price = loband2
                        position = get_long_position('green', i, entry_price, amount, tp_price, sl_price, trailing_tp, tp_line, tp_mode)

                    else:  # short
                        # SL на красной линии
                        sl_price = upband2
                        position = get_short_position('green', i, entry_price, amount, tp_price, sl_price, trailing_tp, tp_line, tp_mode)

                    continue

        position, trades = exit_from_position('green', position, upband1, meanline, loband1, current_high, current_low, trailing_tp_distance, slippage, commission, trades, i)

    # ========== ПРИНУДИТЕЛЬНОЕ ЗАКРЫТИЕ ==========
    if position is not None and force_close_at_end:
        trades.append(force_close_position(position, df_back, slippage, commission, capital_per_trade))

    return trades

In [ ]:
%load_ext autoreload
%autoreload

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import SRC.LIBRARIES.new_indicator_plot_utils as nipu

# ========== НАСТРОЙКИ ОТОБРАЖЕНИЯ ==========
is_log_scale_by_default = True
candlestick_row = 1

# ========== СОЗДАНИЕ ГРАФИКА ==========
fig_main = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0.03)

fig_main.add_trace(
    go.Candlestick(
        x=df[_KIEV_TIMESTAMP],
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC"
    ),
    row=candlestick_row, col=1
)

nipu.add_mrc(candlestick_row, fig_main, df)

# ========== ЗАПУСК БЭКТЕСТА ==========
all_trades = []

# Красная стратегия (всегда активна)
red_trades = backtest_red_cross(
    df=df,
    df_counter=df_counter,
    commission=COMMISSION_RATE,
    capital_per_trade=CAPITAL_PER_TRADE,
    slippage=SLIPPAGE,
    tp_level=RED_TP_LEVEL,
    tp_mode=RED_TP_MODE,
    trailing_tp=TRAILING_TP,
    trailing_tp_distance=TRAILING_TP_DISTANCE,
    sl_ratio=RED_SL_RATIO,
    require_green_touch_after_sl=RED_REQUIRE_GREEN_TOUCH_AFTER_SL,
    min_bars_outside=RED_MIN_BARS_OUTSIDE,
    stoch_filter=RED_STOCH_FILTER,
    stoch_series=RED_STOCH_SERIES,
    stoch_oversold=RED_STOCH_OVERSOLD,
    stoch_overbought=RED_STOCH_OVERBOUGHT,
    force_close_at_end=FORCE_CLOSE_AT_END
)
all_trades.extend(red_trades)
print(f"Красная стратегия: {len(red_trades)} сделок")

# Зелёная стратегия (опционально)
if ENABLE_GREEN_STRATEGY:
    green_trades = backtest_green_cross(
        df=df,
        df_counter=df_counter,
        commission=COMMISSION_RATE,
        capital_per_trade=CAPITAL_PER_TRADE,
        slippage=SLIPPAGE,
        tp_level=GREEN_TP_LEVEL,
        tp_mode=GREEN_TP_MODE,
        trailing_tp=TRAILING_TP,
        trailing_tp_distance=TRAILING_TP_DISTANCE,
        require_yellow_touch_after_sl=GREEN_REQUIRE_YELLOW_TOUCH_AFTER_SL,
        min_bars_outside=GREEN_MIN_BARS_OUTSIDE,
        stoch_filter=GREEN_STOCH_FILTER,
        stoch_series=GREEN_STOCH_SERIES,
        stoch_oversold=GREEN_STOCH_OVERSOLD,
        stoch_overbought=GREEN_STOCH_OVERBOUGHT,
        force_close_at_end=FORCE_CLOSE_AT_END
    )
    all_trades.extend(green_trades)
    print(f"Зелёная стратегия: {len(green_trades)} сделок")

# Общая статистика
metrics = calculate_metrics(all_trades, CAPITAL_PER_TRADE)

# ========== ВЫВОД РЕЗУЛЬТАТОВ ==========
print(f"\nРезультаты стратегии (обратное пересечение красной линии):")
print(f"{SYM} {DISCRETIZATION} | {DISPLAY_START_DATE_STR} - {DISPLAY_END_DATE_STR}")
print(f"RED_TP_LEVEL: {RED_TP_LEVEL}")
print(f"RED_TP_MODE: {RED_TP_MODE}")
print(f"RED_SL_RATIO: 1:{RED_SL_RATIO}")
print(f"Требовать касание зелёной после выхода: {RED_REQUIRE_GREEN_TOUCH_AFTER_SL}")
print(f"Трейлинг-ТП: включён={TRAILING_TP}, отступ={TRAILING_TP_DISTANCE}%")
print(f"RED_STOCH_FILTER: {RED_STOCH_FILTER} (RED_STOCH_SERIES={RED_STOCH_SERIES}, RED_STOCH_OVERSOLD={RED_STOCH_OVERSOLD}, RED_STOCH_OVERBOUGHT={RED_STOCH_OVERBOUGHT})")

if ENABLE_GREEN_STRATEGY:
    print(f"\nЗелёная стратегия: TP={GREEN_TP_LEVEL}({GREEN_TP_MODE}), мин.свечей за зелёной={GREEN_MIN_BARS_OUTSIDE}")

print("-" * 40)

for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

# ========== НАСТРОЙКИ ГРАФИКА ==========
price_log_scale_value = "log"
price_linear_scale_value = "linear"
price_log_scale_title = "Price (log scale)"
price_linear_scale_title = "Price (linear scale)"

fig_main.update_layout(
    title=f"🐵 {DISCRETIZATION} | TP={RED_TP_LEVEL}({RED_TP_MODE}) | SL=1:{RED_SL_RATIO}",
    xaxis_rangeslider_visible=False,
    yaxis1_type=price_log_scale_value if is_log_scale_by_default else price_linear_scale_value,
    yaxis1_title=price_log_scale_title if is_log_scale_by_default else price_linear_scale_title,
    height=1200,
    bargap=0,
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            active=1 if is_log_scale_by_default else 0,
            x=0.315,
            y=1.073,
            buttons=[
                dict(
                    label="Linear",
                    method="relayout",
                    args=[{"yaxis.type": price_linear_scale_value, "yaxis.title.text": price_linear_scale_title}]
                ),
                dict(
                    label="Log",
                    method="relayout",
                    args=[{"yaxis.type": price_log_scale_value, "yaxis.title.text": price_log_scale_title}]
                )
            ],
            font=dict(color="red", size=12, family="Arial"),
            bgcolor="rgba(0, 0, 0, 0)",
            bordercolor="red",
            borderwidth=1
        )
    ]
)

# ========== МАРКЕРЫ СДЕЛОК ==========

# Собираем данные для маркеров
trades_df = pd.DataFrame(all_trades) if all_trades else pd.DataFrame()

def go_scatter(type_col, type_name, group_idx, group_price, name, symbol, size, color, opacity):
    for strategy, group in trades_df[trades_df[type_col] == type_name].groupby('strategy'):
        fig_main.add_trace(
            go.Scatter(
                x=df[_KIEV_TIMESTAMP].iloc[group[group_idx]],
                y=group[group_price],
                mode='markers',
                name=f'{name} ({strategy})',
                marker=dict(
                    symbol=symbol,
                    size=size,
                    color=color,
                    opacity=opacity,
                    line=dict(width=2, color='darkred' if strategy == 'red' else 'darkgreen')
                )
            ),
            row=candlestick_row, col=1
        )

if len(trades_df) > 0:
    go_scatter('type', 'long', 'entry_idx', 'entry_price', 'Long Entry', 'triangle-up', 10, 'green', 1)
    go_scatter('type', 'short', 'entry_idx', 'entry_price', 'Short Entry', 'triangle-down', 10, 'red', 1)
    go_scatter('exit_type', 'tp', 'exit_idx', 'exit_price', 'Take Profit', 'circle', 10, 'lime', 0.7)
    go_scatter('exit_type', 'tp_trailing', 'exit_idx', 'exit_price', 'TP (Trailing)', 'diamond', 12, 'lightgreen', 0.7)
    go_scatter('exit_type', 'sl', 'exit_idx', 'exit_price', 'Stop Loss', 'x', 12, 'red', 0.7)

fig_main.show()

os.system('afplay /System/Library/Sounds/Glass.aiff')